<a href="https://colab.research.google.com/github/nikolaicarrion/2026_Intro_Python/blob/main/FEEBACK_NIKOLAI_CARRION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.4-matplotlib-and-xarray-practice-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# Feedback Nikolai Carrion

One of the most challenging exercise was exercise 14 of the chapter 1.4 about plots and xarrays. I will gradually break down the problem encoutered and show you how I was able to solve this exercise.

# General Comment

Xarray is a completly new concept that was introduced during this course and was not only hard to visualise what was a DataArray, but it was particularly hard to manipulate the data. During this exercice, eventhough it's broken in several parts, I still faced challenges, especially in the conceptual parts.

Though the synthaxe, most of the time, was somewhat not a problem, but as said above, the selection and the manipulation of the data inside the initial Xarray Dataset was really challenging in understanding, which later caused logical errors.

In this feedback, I will breakdown the errors encoutered and the thought process to solve them.

## Exercise 14: Select, reduce, and subtract

Build the small Dataset below, then:

1. Print the spatial mean of the first day (by position) and the time mean at each longitude
   along 47.0° N (by label).
2. Select the grid point nearest 46.8° N, 6.3° E, and print its coordinates and its ten values.
3. Subtract each day's spatial mean — the mean over `lat` and `lon` — from the field. Print the
   dims of the result, and its spatial mean on the first day, which should be zero up to rounding.

```python
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})
```

# Part 1

The first step was to study the given code, which contains the Xarray Dataset and the variables which were its building blocks.

The first challenge was to read and understand the Dataset which lead to my first error : using ds["time"] instead of ds["t2m"]. The fact that I didn't get an error was missleading, though the answer was obviously incorrect, since it gave a date instead of a temperature.

Let's break it down:

## Reading and understanding a Dataset

{"t2m": (("time", "lat", "lon"), t)

This line creates a new variable "t2m" that associates our temperature values "t" with the dimensions "time", "lat" and "lon".

The "dimension" part was really confusing.



In [23]:
# Provided data

import numpy as np
import xarray as xr

rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})

In [24]:
# Part 1
spatial_mean_day1 = ds["time"].isel(time = 0).mean().item()
print(f"The spatial mean for day 1 is {spatial_mean_day1} °C")

The spatial mean for day 1 is 2024-01-01 00:00:00 °C


This first line of code gives a date, eventhough I was looking on selecting the temperature. By doing, ds["time"], I'm selecting the time variable, thinking that each time is directly associated with a temperature, almost as a dictionary.

To correct this line, I simply had to change "time" with "t2m", which was defined ealier in the xr.Dataset.

As mentionned before, it was difficult to understand how a Dataset is built.
"t2m":(("time","lat","lon"), t) mean that you are creating a new variable "t2m", where you then assigne time, lat and lon for every temperature in t. The actual time, lat and lon is then assigned in the line below with coords.

In [25]:
# Day 1
spatial_mean_day1 = ds["t2m"].isel(time =0).mean().item()
print(f"The spatial mean for day 1 is {round(spatial_mean_day1,2)} °C")


# Latitude 47 °N
spatial_mean_day1_lon = ds["t2m"].sel(lat = 47.0).mean().item()
print(f"The spatial mean for lat 47 °N {round(spatial_mean_day1_lon,2)} °C")

The spatial mean for day 1 is 5.07 °C
The spatial mean for lat 47 °N 5.22 °C


In [17]:
# Part 2
point = ds["t2m"].sel((lat == 46.8) & (lon == 6.3) , method= "nearest")
print("Latitude:", point["lat"].item())
print("Longitude:", point["lon"].item())
print("Values:", point.values)

ValueError: operands could not be broadcast together with shapes (3,) (4,) 

https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sel.html

In [18]:
# Correction part 2

point = ds["t2m"].sel(lat = 46.8, lon = 6.3, method= "nearest")

print("Latitude:", point["lat"].item())
print("Longitude:", point["lon"].item())
print("Values:", point.values)

Latitude: 47.0
Longitude: 6.5
Values: [ 1.20373559  9.09939041  6.62253675  4.05823156  7.08812817 -1.61052964
  3.10041773  2.79355012  1.15881817  7.17828137]


For part 3, the terme "spatial mean" wasn't very clear at first and the mean function was also

In [21]:
# Part 3
result = ds["t2m"] - ds["t2m"].mean(dim=("time"))


('time', 'lat', 'lon')


TypeError: type DataArray doesn't define __round__ method

In [22]:
# Part 3
result = ds["t2m"] - ds["t2m"].mean(dim=("lat", "lon"))

print()